In [2]:
import os
from pathlib import Path

DC_DIR = Path("/home/tjdgns1502/runs/deepchem")
DC_DIR.mkdir(parents=True, exist_ok=True)

os.environ["DEEPCHEM_DATA_DIR"] = str(DC_DIR)
print("DEEPCHEM_DATA_DIR =", os.environ["DEEPCHEM_DATA_DIR"])

DEEPCHEM_DATA_DIR = /home/tjdgns1502/runs/deepchem


In [3]:
# 0) Environment checks (run once)
import os, sys, math, json, random, csv, time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterable, Optional, Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, IterableDataset

# Optional dependencies (only needed if you use HuggingFace local cache download helpers)
try:
    from huggingface_hub import snapshot_download
    _HAS_HF_HUB = True
except Exception:
    _HAS_HF_HUB = False

from transformers import (
    RobertaTokenizer,
    RobertaConfig,
    RobertaForMaskedLM,
    DataCollatorForLanguageModeling,
)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


torch: 2.10.0+cu128
cuda available: True
gpu: NVIDIA A100 80GB PCIe


In [4]:
# 1) Repro-friendly paths & config
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def _default_root() -> Path:
    return Path.cwd()

@dataclass
class SaeConfig:
    # Model / tokenizer
    model_name: str = "seyonec/ChemBERTa-zinc-base-v1"
    local_only: bool = False

    # Data for MLM stream used to harvest activations (SMILES, 1 per line)
    mlm_data_path: Path = _default_root() / "data" / "100k_rndm_zinc_drugs_clean.txt"
    max_len: int = 128
    mlm_batch_size: int = 8
    mlm_mask_prob: float = 0.15

    # Which layer to extract from (0-indexed encoder layer id)
    layer: int = 5

    # Activation extraction (token-level) → saved as chunks on disk
    runs_dir: Path = _default_root() / "runs" / "sae"
    acts_dir: Path = runs_dir / "acts"
    chunk_size_tokens: int = 20000  # number of (masked) tokens per saved .pt file
    save_dtype: str = "float16"

    # SAE hyperparameters (OpenAI-style Top‑K SAE)
    d_model: Optional[int] = None     # inferred after first forward
    n_latents: int = 4096
    topk: int = 32
    sae_lr: float = 1e-4
    sae_batch_tokens: int = 2048      # number of token activations per optimization step
    sae_steps: int = 50_000           # prefer steps over epochs (streaming)
    weight_decay: float = 0.0
    grad_clip: float = 1.0

    # Loss terms
    l1_weight: float = 0.0            # optional additional L1 on latents
    recon_loss: str = "mse"           # "mse" or "huber"
    aux_k: int = 0                    # set >0 to use a "dead-latent" auxiliary reconstruction trick

    # Checkpointing / resume
    ckpt_dir: Path = runs_dir / "checkpoints"
    log_csv: Path = runs_dir / "train_log.csv"
    ckpt_every: int = 2000

    # Downstream (optional; requires your existing molnet loader code)
    downstream_tasks: Tuple[str, ...] = ("bbbp", "bace_classification", "clintox")

cfg = SaeConfig()
cfg.runs_dir.mkdir(parents=True, exist_ok=True)
cfg.acts_dir.mkdir(parents=True, exist_ok=True)
cfg.ckpt_dir.mkdir(parents=True, exist_ok=True)

print(cfg)


SaeConfig(model_name='seyonec/ChemBERTa-zinc-base-v1', local_only=False, mlm_data_path=PosixPath('/home/tjdgns1502/data/100k_rndm_zinc_drugs_clean.txt'), max_len=128, mlm_batch_size=8, mlm_mask_prob=0.15, layer=5, runs_dir=PosixPath('/home/tjdgns1502/runs/sae'), acts_dir=PosixPath('/home/tjdgns1502/runs/sae/acts'), chunk_size_tokens=20000, save_dtype='float16', d_model=None, n_latents=4096, topk=32, sae_lr=0.0001, sae_batch_tokens=2048, sae_steps=50000, weight_decay=0.0, grad_clip=1.0, l1_weight=0.0, recon_loss='mse', aux_k=0, ckpt_dir=PosixPath('/home/tjdgns1502/runs/sae/checkpoints'), log_csv=PosixPath('/home/tjdgns1502/runs/sae/train_log.csv'), ckpt_every=2000, downstream_tasks=('bbbp', 'bace_classification', 'clintox'))


In [5]:
# 2) Load ChemBERTa (weights from local cache if available)

def load_state_dict_from_hf(model_name: str, local_only: bool = True) -> Dict[str, torch.Tensor]:
    if not _HAS_HF_HUB:
        raise RuntimeError("huggingface_hub is not installed. `pip install huggingface_hub` or set local_only=True and load manually.")
    snapshot_dir = snapshot_download(repo_id=model_name, local_files_only=local_only)
    snapshot_dir = Path(snapshot_dir)
    safetensors_path = snapshot_dir / "model.safetensors"
    bin_path = snapshot_dir / "pytorch_model.bin"
    if safetensors_path.exists():
        from safetensors.torch import load_file
        return load_file(str(safetensors_path))
    return torch.load(str(bin_path), map_location="cpu")

def load_config_from_hf(model_name: str, local_only: bool = True) -> RobertaConfig:
    if not _HAS_HF_HUB:
        # Fallback: transformers can still load config via from_pretrained if files are present locally.
        return RobertaConfig.from_pretrained(model_name, local_files_only=local_only)
    snapshot_dir = snapshot_download(repo_id=model_name, local_files_only=local_only)
    cfg_path = Path(snapshot_dir) / "config.json"
    with open(cfg_path) as f:
        cfg_dict = json.load(f)
    return RobertaConfig(**cfg_dict)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(42)

tokenizer = RobertaTokenizer.from_pretrained(cfg.model_name, local_files_only=cfg.local_only)

# Build model similarly to your teammate notebook: instantiate by config then inject state_dict.
roberta_cfg = load_config_from_hf(cfg.model_name, local_only=cfg.local_only)
model = RobertaForMaskedLM(roberta_cfg)
model.tie_weights()

if cfg.local_only and not _HAS_HF_HUB:
    # If no hf_hub helper, use transformers direct load (works when cached)
    model = RobertaForMaskedLM.from_pretrained(cfg.model_name, local_files_only=True).to(device)
else:
    state_dict = load_state_dict_from_hf(cfg.model_name, local_only=cfg.local_only) if _HAS_HF_HUB else None
    if state_dict is not None:
        model.load_state_dict(state_dict, strict=False)
    model = model.to(device)

model.eval()
print("Loaded model hidden_size:", roberta_cfg.hidden_size)


Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Loaded model hidden_size: 768


In [6]:
# 3) MLM dataloader for harvesting activations (HF canonicalized ZINC 기반)

from torch.utils.data import DataLoader, IterableDataset
from transformers import DataCollatorForLanguageModeling, RobertaTokenizer
from datasets import load_dataset

class HFSmilesIterableDataset(IterableDataset):
    """
    HuggingFace datasets에서 SMILES 컬럼을 스트리밍으로 읽어서
    tokenizer에 바로 넣을 수 있는 dict(input_ids, attention_mask)를 yield.
    """
    def __init__(
        self,
        dataset_name: str,
        split: str,
        tokenizer: RobertaTokenizer,
        smiles_col: str = "smiles",
        max_len: int = 128,
        streaming: bool = True,
        seed: int = 42,
        shuffle_buffer: int = 10_000,
        limit: int | None = None, 
    ):
        super().__init__()
        self.dataset_name = dataset_name
        self.split = split
        self.tokenizer = tokenizer
        self.smiles_col = smiles_col
        self.max_len = max_len
        self.streaming = streaming
        self.seed = seed
        self.shuffle_buffer = shuffle_buffer
        self.limit = limit

    def __iter__(self):
        ds = load_dataset(self.dataset_name, split=self.split, streaming=self.streaming)

        # streaming일 때 shuffle은 buffer 기반으로 동작
        if self.streaming:
            ds = ds.shuffle(seed=self.seed, buffer_size=self.shuffle_buffer)

        n = 0
        for ex in ds:
            s = ex.get(self.smiles_col, None)
            if not s:
                continue
            s = str(s).strip()
            if not s:
                continue

            enc = self.tokenizer(
                s,
                truncation=True,
                padding="max_length",
                max_length=self.max_len,
                return_tensors="pt",
            )
            yield {k: v.squeeze(0) for k, v in enc.items()}

            n += 1
            if self.limit is not None and n >= self.limit:
                break


def prepare_mlm_loader(cfg, tokenizer: RobertaTokenizer) -> DataLoader:
    ds = HFSmilesIterableDataset(
        dataset_name=cfg.hf_dataset_name,
        split=cfg.hf_split,
        tokenizer=tokenizer,
        smiles_col=cfg.hf_smiles_col,
        max_len=cfg.max_len,
        streaming=True,
        seed=getattr(cfg, "seed", 42),
        shuffle_buffer=getattr(cfg, "shuffle_buffer", 10_000),
        limit=getattr(cfg, "limit", None),
    )

    collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer, mlm=True, mlm_probability=cfg.mlm_mask_prob
    )
    return DataLoader(ds, batch_size=cfg.mlm_batch_size, collate_fn=collator)


# 사용 예시 (cfg에 필드 추가)
cfg.hf_dataset_name = "sagawa/ZINC-canonicalized"
cfg.hf_split = "train"
cfg.hf_smiles_col = "smiles"
cfg.shuffle_buffer = 10_000
cfg.seed = 42

mlm_loader = prepare_mlm_loader(cfg, tokenizer)
batch = next(iter(mlm_loader))
print({k: v.shape for k, v in batch.items()})


{'input_ids': torch.Size([8, 128]), 'attention_mask': torch.Size([8, 128]), 'labels': torch.Size([8, 128])}


In [7]:
# 4) Activation extraction (token-level) via forward hooks
# This replaces the teammate's custom-RoBERTa "return_attn_outputs" path with a more portable hook method.
#
# We extract a tensor shaped [B, T, H] from the chosen layer and then flatten only valid tokens:
# flat = acts[attention_mask.bool()] -> [N_tokens, H]

class HookCatcher:
    def __init__(self):
        self.value = None
    def __call__(self, module, inp, out):
        self.value = out

def _write_chunk(layer_dir: Path, chunk_idx: int, tensor: torch.Tensor) -> Path:
    layer_dir.mkdir(parents=True, exist_ok=True)
    path = layer_dir / f"chunk_{chunk_idx:05d}.pt"
    torch.save(tensor, path)
    return path

def save_meta(path: Path, meta: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

@torch.no_grad()
def extract_activations_to_chunks(
    cfg: SaeConfig,
    model: RobertaForMaskedLM,
    loader: DataLoader,
    device: torch.device,
    which: str = "attention_output",
) -> None:
    model.eval()
    layer_dir = cfg.acts_dir / f"layer_{cfg.layer}_{which}"
    layer_dir.mkdir(parents=True, exist_ok=True)

    # pick module to hook (HuggingFace RoBERTa internals)
    layer_mod = model.roberta.encoder.layer[cfg.layer]
    if which == "attention_output":
        target = layer_mod.attention.output
    elif which == "mlp_output":
        target = layer_mod.output
    elif which == "residual_stream":
        target = layer_mod
    else:
        raise ValueError("which must be one of: attention_output, mlp_output, residual_stream")

    catcher = HookCatcher()
    handle = target.register_forward_hook(catcher)

    chunk_idx = 0
    buffered = []
    buffered_tokens = 0
    total_tokens = 0
    chunk_paths = []

    for step, batch in enumerate(loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # forward pass (we don't need logits)
        _ = model.roberta(input_ids=input_ids, attention_mask=attention_mask)

        acts = catcher.value
        # acts can be tuple if hooking layer_mod; normalize to tensor
        if isinstance(acts, (tuple, list)):
            acts = acts[0]
        if acts is None:
            raise RuntimeError("Hook did not capture activations. Check which=... target module.")

        # [B,T,H] expected; if [T,B,H] or others, adjust here
        if acts.dim() != 3:
            raise RuntimeError(f"Expected 3D activations [B,T,H], got shape {tuple(acts.shape)}")

        flat = acts[attention_mask.bool()].detach().cpu()
        if cfg.save_dtype == "float16":
            flat = flat.to(torch.float16)
        elif cfg.save_dtype == "float32":
            flat = flat.to(torch.float32)

        buffered.append(flat)
        buffered_tokens += flat.shape[0]
        total_tokens += flat.shape[0]

        if buffered_tokens >= cfg.chunk_size_tokens:
            chunk = torch.cat(buffered, dim=0)
            chunk_paths.append(_write_chunk(layer_dir, chunk_idx, chunk))
            chunk_idx += 1
            buffered = []
            buffered_tokens = 0

        if (step + 1) % 100 == 0:
            print(f"[extract] step={step+1} total_tokens={total_tokens} chunks={len(chunk_paths)}")

    if buffered:
        chunk = torch.cat(buffered, dim=0)
        chunk_paths.append(_write_chunk(layer_dir, chunk_idx, chunk))

    meta = {
        "model_name": cfg.model_name,
        "layer": cfg.layer,
        "which": which,
        "hidden_size": acts.shape[-1],
        "num_tokens": int(total_tokens),
        "num_chunks": len(chunk_paths),
        "chunk_size_tokens": cfg.chunk_size_tokens,
        "dtype": cfg.save_dtype,
        "mlm_data_path": str(cfg.mlm_data_path),
        "max_len": cfg.max_len,
        "mlm_batch_size": cfg.mlm_batch_size,
    }
    save_meta(layer_dir / "meta.json", meta)
    handle.remove()
    print("Saved:", layer_dir / "meta.json")

extract_activations_to_chunks(cfg, model, mlm_loader, device, which="attention_output")


In [8]:
# 5) Streaming token-activation dataset (matches teammate chunk format)
def list_chunks(layer_dir: Path) -> List[Path]:
    return sorted(layer_dir.glob("chunk_*.pt"))

class ActivationChunkDataset(IterableDataset):
    """Streams token activations from saved chunk files, yielding batches of shape [B, d_model]."""
    def __init__(self, chunk_paths: List[Path], batch_size: int, shuffle: bool, seed: int):
        super().__init__()
        self.chunk_paths = list(chunk_paths)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.seed = seed

    def _iter_files(self):
        paths = self.chunk_paths
        if self.shuffle:
            rng = random.Random(self.seed + int(time.time()))
            paths = paths.copy()
            rng.shuffle(paths)
        for p in paths:
            yield p

    def __iter__(self):
        for p in self._iter_files():
            x = torch.load(p, map_location="cpu")  # [N, d_model]
            if self.shuffle:
                idx = torch.randperm(x.shape[0])
                x = x[idx]
            n = x.shape[0]
            for i in range(0, n, self.batch_size):
                yield x[i:i+self.batch_size]



In [9]:
# 6) Top‑K Sparse Autoencoder (portable implementation)
#
# Encoder: h = TopK(W_enc x + b_enc)  (optionally also L1 penalty)
# Decoder: x_hat = W_dec h + b_dec
#
# Practical tricks from modern SAE work:
# - normalize inputs (norm to sqrt(d_model))
# - constrain decoder column norms (unit-norm) to avoid trivial scaling
# - Top‑K sparsity to keep exactly k active latents per token

class TopK(nn.Module):
    def __init__(self, k: int):
        super().__init__()
        self.k = int(k)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, n_latents]
        if self.k <= 0 or self.k >= x.shape[-1]:
            return x
        vals, idx = torch.topk(x, self.k, dim=-1)
        out = torch.zeros_like(x)
        out.scatter_(-1, idx, vals)
        return out

class TopKSAE(nn.Module):
    def __init__(self, n_inputs: int, n_latents: int, k: int, normalize_inputs: bool = True):
        super().__init__()
        self.n_inputs = n_inputs
        self.n_latents = n_latents
        self.k = k
        self.normalize_inputs = normalize_inputs

        # Encoder / decoder
        self.W_enc = nn.Parameter(torch.empty(n_latents, n_inputs))
        self.b_enc = nn.Parameter(torch.zeros(n_latents))
        self.W_dec = nn.Parameter(torch.empty(n_inputs, n_latents))
        self.b_dec = nn.Parameter(torch.zeros(n_inputs))

        nn.init.kaiming_uniform_(self.W_enc, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.W_dec, a=math.sqrt(5))

        self.act = TopK(k)

    def _normalize_x(self, x: torch.Tensor) -> torch.Tensor:
        if not self.normalize_inputs:
            return x
        # scale to have norm ~ sqrt(d)
        d = x.shape[-1]
        eps = 1e-6
        norm = x.norm(dim=-1, keepdim=True).clamp(min=eps)
        return x * (math.sqrt(d) / norm)

    @torch.no_grad()
    def _renorm_decoder(self):
        # unit-norm decoder columns (common SAE stabilization trick)
        eps = 1e-6
        col_norm = self.W_dec.norm(dim=0, keepdim=True).clamp(min=eps)
        self.W_dec.div_(col_norm)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        x = self._normalize_x(x)
        pre = F.linear(x, self.W_enc, self.b_enc)  # [B, n_latents]
        h = self.act(pre)
        return h, pre

    def decode(self, h: torch.Tensor) -> torch.Tensor:
        x_hat = F.linear(h, self.W_dec, self.b_dec)  # [B, n_inputs]
        return x_hat

    def forward(self, x: torch.Tensor):
        h, pre = self.encode(x)
        x_hat = self.decode(h)
        return x_hat, h, pre

def recon_loss(x_hat: torch.Tensor, x: torch.Tensor, kind: str = "mse") -> torch.Tensor:
    if kind == "mse":
        return F.mse_loss(x_hat, x)
    if kind == "huber":
        return F.smooth_l1_loss(x_hat, x)
    raise ValueError("kind must be mse or huber")

def sae_loss(
    x_hat: torch.Tensor,
    x: torch.Tensor,
    h: torch.Tensor,
    pre: torch.Tensor,
    l1_weight: float = 0.0,
    kind: str = "mse",
) -> torch.Tensor:
    loss = recon_loss(x_hat, x, kind=kind)
    if l1_weight and l1_weight > 0:
        # optional extra sparsity pressure
        loss = loss + l1_weight * h.abs().mean()
    return loss



In [10]:
# 7) SAE training loop with checkpoint resume (mirrors teammate notebook)
def latest_checkpoint(layer_ckpt_dir: Path) -> Optional[Path]:
    latest = layer_ckpt_dir / "latest.pt"
    if latest.exists():
        return latest
    ckpts = sorted(layer_ckpt_dir.glob("checkpoint_step_*.pt"))
    return ckpts[-1] if ckpts else None

def save_checkpoint(path: Path, state: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(state, path)

def append_csv_row(path: Path, row: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    write_header = not path.exists()
    with open(path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if write_header:
            writer.writeheader()
        writer.writerow(row)

from dataclasses import asdict
from pathlib import Path

def cfg_to_jsonable(d):
    out = {}
    for k, v in d.items():
        if isinstance(v, Path):
            out[k] = str(v)
        else:
            out[k] = v
    return out

def train_sae(
    cfg: SaeConfig,
    which: str = "attention_output",
    resume: bool = True,
) -> TopKSAE:
    layer_dir = cfg.acts_dir / f"layer_{cfg.layer}_{which}"
    chunk_paths = list_chunks(layer_dir)
    if not chunk_paths:
        raise FileNotFoundError(f"No activation chunks found in {layer_dir}. Run extraction first.")
    meta_path = layer_dir / "meta.json"
    meta = json.load(open(meta_path, encoding="utf-8")) if meta_path.exists() else {}
    d_model = meta.get("hidden_size", None)
    if d_model is None:
        d_model = torch.load(chunk_paths[0], map_location="cpu").shape[1]
    cfg.d_model = int(d_model)

    ds = ActivationChunkDataset(
        chunk_paths=chunk_paths,
        batch_size=cfg.sae_batch_tokens,
        shuffle=True,
        seed=cfg.seed,
    )

    ae = TopKSAE(
        n_inputs=cfg.d_model,
        n_latents=cfg.n_latents,
        k=cfg.topk,
        normalize_inputs=True,
    ).to(device)

    opt = torch.optim.AdamW(ae.parameters(), lr=cfg.sae_lr, weight_decay=cfg.weight_decay)

    layer_ckpt_dir = cfg.ckpt_dir / f"layer_{cfg.layer}_{which}"
    layer_ckpt_dir.mkdir(parents=True, exist_ok=True)

    start_step = 0
    if resume:
        ckpt = latest_checkpoint(layer_ckpt_dir)
        if ckpt is not None:
            state = torch.load(ckpt, map_location=device, weights_only=False)
            ae.load_state_dict(state["model"])
            opt.load_state_dict(state["optimizer"])
            start_step = int(state["step"])
            print("Resumed from", ckpt, "step", start_step)

    ae.train()
    t0 = time.time()
    step = start_step

    # IterableDataset yields batches forever only if we loop; we re-create iterator each epoch-like pass.
    while step < cfg.sae_steps:
        for batch in ds:
            batch = batch.to(device).float()  # [B, d_model]
            x_hat, h, pre = ae(batch)
            loss = sae_loss(x_hat, batch, h, pre, l1_weight=cfg.l1_weight, kind=cfg.recon_loss)

            loss.backward()
            if cfg.grad_clip and cfg.grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(ae.parameters(), cfg.grad_clip)
            opt.step()
            opt.zero_grad(set_to_none=True)

            # decoder renorm (stabilization)
            ae._renorm_decoder()

            step += 1
            if step % 200 == 0:
                dt = time.time() - t0
                sparsity = (h != 0).float().mean().item()
                row = {
                    "step": step,
                    "loss": float(loss.item()),
                    "active_frac": float(sparsity),
                    "layer": cfg.layer,
                    "which": which,
                    "n_latents": cfg.n_latents,
                    "topk": cfg.topk,
                    "lr": cfg.sae_lr,
                    "elapsed_sec": dt,
                }
                append_csv_row(cfg.log_csv, row)
                print(row)

            if step % cfg.ckpt_every == 0:
                state = {
                    "model": ae.state_dict(),
                    "optimizer": opt.state_dict(),
                    "step": step,
                    "cfg": cfg_to_jsonable(asdict(cfg)),
                }
                save_checkpoint(layer_ckpt_dir / f"checkpoint_step_{step:07d}.pt", state)
                save_checkpoint(layer_ckpt_dir / "latest.pt", state)

            if step >= cfg.sae_steps:
                break

    # final save
    state = {"model": ae.state_dict(), "optimizer": opt.state_dict(), "step": step, "cfg": asdict(cfg)}
    save_checkpoint(layer_ckpt_dir / "latest.pt", state)
    print("Done. Saved latest checkpoint to", layer_ckpt_dir / "latest.pt")
    return ae

# Example:
# ae = train_sae(cfg, which="attention_output", resume=False)


In [11]:
import sys
sys.path.append("chemberta_SAE/code/bert-loves-chemistry")

In [17]:
# --- HF MolNet loader (DeepChem 우회) ---
from datasets import load_dataset

def load_task_df(task: str):
    # MolNet HF dataset ids (zpn/*)
    if task == "bbbp":
        ds = load_dataset("zpn/bbbp")
        label_cols = ["target"]        # zpn/bbbp: smiles, selfies, target
        smiles_col = "smiles"

    elif task in ["bace", "bace_classification"]:
        ds = load_dataset("zpn/bace_classification")
        smiles_col = "smiles"
        label_cols = None

    elif task == "clintox":
        ds = load_dataset("zpn/clintox")
        smiles_col = "smiles"
        label_cols = None 

    else:
        raise ValueError(f"Unknown task: {task}")

    train_df = ds["train"].to_pandas()
    valid_df = ds["validation"].to_pandas() if "validation" in ds else ds["valid"].to_pandas()
    test_df  = ds["test"].to_pandas()

    if label_cols is None:
        drop_cols = {smiles_col, "selfies", "mol_id", "id", "ids"}
        candidates = [c for c in train_df.columns if c not in drop_cols]
        if not candidates:
            raise ValueError(f"Could not infer label columns for {task}. columns={list(train_df.columns)}")
        label_cols = candidates

    return label_cols, (train_df, valid_df, test_df)

In [19]:
# 8) Downstream probe (optional; uses your existing MolNet loader if available)
# This mirrors sae_experiment.ipynb. If your repo has `chemberta.utils.molnet_dataloader`,
# it will work; otherwise you can replace with your own CSV loader.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def _try_import_molnet():
    try:
        from chemberta.utils.molnet_dataloader import load_molnet_dataset
        return load_molnet_dataset
    except Exception as e:
        print("MolNet loader not available:", e)
        return None

load_molnet_dataset = _try_import_molnet()

class SmilesClassificationDataset(Dataset):
    def __init__(self, df, tokenizer, label_cols, max_len: int = 128, smiles_col: str = "smiles"):
        if smiles_col not in df.columns:
            raise KeyError(f"smiles_col='{smiles_col}' not in df.columns={list(df.columns)}")

        self.texts = df[smiles_col].astype(str).tolist()
        self.labels = df[label_cols].values
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

@torch.no_grad()
def compute_latent_features(
    model: RobertaForMaskedLM,
    ae: TopKSAE,
    dataloader: DataLoader,
    layer: int,
    which: str,
    device: torch.device,
):
    model.eval()
    ae.eval()

    layer_mod = model.roberta.encoder.layer[layer]
    if which == "attention_output":
        target = layer_mod.attention.output
    elif which == "mlp_output":
        target = layer_mod.output
    elif which == "residual_stream":
        target = layer_mod
    else:
        raise ValueError("which must be attention_output / mlp_output / residual_stream")

    catcher = HookCatcher()
    handle = target.register_forward_hook(catcher)

    feats, labels = [], []
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        y = batch["labels"].cpu()

        _ = model.roberta(input_ids=input_ids, attention_mask=attention_mask)
        acts = catcher.value
        if isinstance(acts, (tuple, list)):
            acts = acts[0]
        flat = acts.reshape(-1, acts.shape[-1])
        h, _ = ae.encode(flat)  # [B*T, n_latents]
        h = h.reshape(acts.shape[0], acts.shape[1], -1)

        mask = attention_mask.unsqueeze(-1).float()
        pooled = (h * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1.0)

        feats.append(pooled.cpu())
        labels.append(y)

    handle.remove()
    return torch.cat(feats, dim=0).numpy(), torch.cat(labels, dim=0).numpy()

def eval_multitask_roc_auc(y_true: np.ndarray, y_prob: np.ndarray) -> float:
    # MolNet datasets often use NaN for missing labels in multi-task settings.
    scores = []
    for j in range(y_true.shape[1]):
        yt = y_true[:, j]
        yp = y_prob[:, j]
        mask = ~np.isnan(yt)
        if mask.sum() < 10:
            continue
        # Need both classes present
        if len(np.unique(yt[mask])) < 2:
            continue
        scores.append(roc_auc_score(yt[mask], yp[mask]))
    return float(np.mean(scores)) if scores else float("nan")

def run_downstream_probe(cfg: SaeConfig, ae: TopKSAE, which: str = "attention_output"):
    if load_molnet_dataset is None:
        raise RuntimeError("No MolNet loader found. Provide `chemberta.utils.molnet_dataloader` or replace this section.")

    results = []
    for task in cfg.downstream_tasks:
        label_cols, (train_df, valid_df, test_df) = load_task_df(task)

        train_ds = SmilesClassificationDataset(train_df, tokenizer, label_cols, max_len=cfg.max_len, smiles_col="smiles")
        test_ds  = SmilesClassificationDataset(test_df,  tokenizer, label_cols, max_len=cfg.max_len, smiles_col="smiles")


        train_loader = DataLoader(train_ds, batch_size=32, shuffle=False)
        test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False)

        X_train, y_train = compute_latent_features(model, ae, train_loader, cfg.layer, which, device)
        X_test,  y_test  = compute_latent_features(model, ae, test_loader,  cfg.layer, which, device)

        # One-vs-rest logistic regression for multi-task
        clf = LogisticRegression(max_iter=2000)
        # If multi-task, scikit expects 1D; we fit per-task
        probs = np.zeros_like(y_test, dtype=np.float32)
        for j in range(y_train.shape[1]):
            yt = y_train[:, j]
            mask = ~np.isnan(yt)
            if mask.sum() < 10 or len(np.unique(yt[mask])) < 2:
                probs[:, j] = np.nan
                continue
            clf_j = LogisticRegression(max_iter=2000)
            clf_j.fit(X_train[mask], yt[mask])
            probs[:, j] = clf_j.predict_proba(X_test)[:, 1]

        auc = eval_multitask_roc_auc(y_test, probs)
        row = {"task": task, "layer": cfg.layer, "which": which, "auc": auc}
        results.append(row)
        print(row)

    return results

# Example:
ae = train_sae(cfg, which="attention_output", resume=True)
results = run_downstream_probe(cfg, ae, which="attention_output")


Resumed from /home/tjdgns1502/runs/sae/checkpoints/layer_5_attention_output/latest.pt step 50000
Done. Saved latest checkpoint to /home/tjdgns1502/runs/sae/checkpoints/layer_5_attention_output/latest.pt
{'task': 'bbbp', 'layer': 5, 'which': 'attention_output', 'auc': 0.7401483765295308}


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-82293fde28860d(…):   0%|          | 0.00/93.1k [00:00<?, ?B/s]

data/validation-00000-of-00001-8b8eb3c29(…):   0%|          | 0.00/18.2k [00:00<?, ?B/s]

...

data/test-00000-of-00001-6e352a5dc26e9a8(…):   0%|          | 0.00/14.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1181 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/148 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/148 [00:00<?, ? examples/s]

{'task': 'clintox', 'layer': 5, 'which': 'attention_output', 'auc': 0.8195652173913043}
